## Final Project - Build an ML Pipeline for ticket fares prediction

## Objective

Replicate the Project for the IBM Data Engineering Certificate with a different dataset. The objective is to clean the dataset and create an ML pipeline to create a model that will predict ticket fares based on relevant columns present in the original dataset (Passenger class, sex, age, No. of siblings aboard, No. of parents/children aboard)

## Dataset
CC0 Titanic dataset available at the following link:
https://www.kaggle.com/datasets/yasserh/titanic-dataset 

## Libraries imports and spark session setup

In [2]:

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder

In [3]:
import findspark
findspark.init()

In [4]:
#sparksession creation
spark = SparkSession.builder.appName("Titanic Project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/12 08:40:44 WARN Utils: Your hostname, EmanueleHp, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/12 08:40:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 08:40:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## ETL

In [5]:
df = spark.read.csv(path="../data/raw/Titanic-Dataset.csv",header=True, inferSchema=True)

In [6]:
#Checking the schema of the dataframe
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [7]:
rowCount = df.count()
print(f"Number of initial rows in the dataframe: {rowCount}")
#Dropping the rows with null values and duplicates
df = df.dropna(subset=["Pclass","Sex","Age", "Sibsp","Parch","Fare"])  # Drop rows where any of the relevant columns have null values
df = df.dropDuplicates()
rowCount = df.count()
print(f"Number of rows in the dataframe after dropping null values and duplicates: {rowCount}")

Number of initial rows in the dataframe: 891
Number of rows in the dataframe after dropping null values and duplicates: 714


In [8]:
#selecting and renaming the relevant the columns for better readability
df= df.select("Pclass", "Sex", "Age", "Sibsp", "Parch", "Fare")
df = df.withColumnRenamed("Pclass", "passengerClass") \
         .withColumnRenamed("Sex", "sex") \
         .withColumnRenamed("Age", "age") \
         .withColumnRenamed("Sibsp", "siblingNumber") \
         .withColumnRenamed("Parch", "parentChildNumber") \
         .withColumnRenamed("Fare", "fare") 


In [9]:
#saving the cleaned dataframe in parquet format
df.write.mode("overwrite").parquet("../data/work/NASA_airfoil_noise_cleaned.parquet")

## Creation of a Machine Learning Pipeline

In [10]:
df = spark.read.parquet("../data/work/NASA_airfoil_noise_cleaned.parquet")
df.printSchema()

root
 |-- passengerClass: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: double (nullable = true)
 |-- siblingNumber: integer (nullable = true)
 |-- parentChildNumber: integer (nullable = true)
 |-- fare: double (nullable = true)



In [11]:
#Defining the pipeline stages for the ML model

#StringIndexer to convert the categorical variable "sex" into numerical format
indexer = StringIndexer(inputCol="sex", outputCol="sexIndex")

#OneHotEncoder to convert the indexed "sex" column into a one-hot encoded vector
encoder = OneHotEncoder(inputCols=["sexIndex"], outputCols=["sexVector"])
#VectorAssembler to combine the feature columns into a single vector column
assembler = VectorAssembler(inputCols=["passengerClass", "age", "siblingNumber", "parentChildNumber", "sexVector"], outputCol="features")
#scaling the features using StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
#LinearRegression model to predict the target variable "fare"
lr = LinearRegression(featuresCol="scaledFeatures", labelCol="fare")

In [12]:
#Defining the pipeline
pipeline= Pipeline(stages=[indexer, encoder, assembler, scaler, lr])

In [13]:
#Splitting the data into training and testing sets
trainingData, testingData = df.randomSplit([0.7, 0.3], seed=42)
print(f"Number of rows in the training data: {trainingData.count()}")
print(f"Number of rows in the testing data: {testingData.count()}")

Number of rows in the training data: 526
Number of rows in the testing data: 188


In [14]:
#Fitting the pipeline to the training data
model = pipeline.fit(trainingData)

26/02/12 08:41:01 WARN Instrumentation: [6beadd1c] regParam is zero, which might cause numerical instability and overfitting.
26/02/12 08:41:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/12 08:41:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


## Model Evaluation


In [15]:
#making predictions on the testing data
predictions = model.transform(testingData)
predictions.show(5)

+--------------+------+----+-------------+-----------------+--------+--------+---------+--------------------+--------------------+-----------------+
|passengerClass|   sex| age|siblingNumber|parentChildNumber|    fare|sexIndex|sexVector|            features|      scaledFeatures|       prediction|
+--------------+------+----+-------------+-----------------+--------+--------+---------+--------------------+--------------------+-----------------+
|             1|female|15.0|            0|                1|211.3375|     1.0|(1,[],[])|[1.0,15.0,0.0,1.0...|[1.20544908118579...|78.64258734786299|
|             1|female|17.0|            1|                0|    57.0|     1.0|(1,[],[])|[1.0,17.0,1.0,0.0...|[1.20544908118579...|73.69758170630871|
|             1|female|18.0|            0|                2|   79.65|     1.0|(1,[],[])|[1.0,18.0,0.0,2.0...|[1.20544908118579...|88.58837862656145|
|             1|female|18.0|            1|                0| 227.525|     1.0|(1,[],[])|[1.0,18.0,1.0,0.0.

In [16]:
#Metrics for evaluating the model
evaluator_rmse = RegressionEvaluator(
    labelCol="Fare", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(
    labelCol="Fare", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(
    labelCol="Fare", predictionCol="prediction", metricName="r2")

print(f"Root Mean Square Error: {evaluator_rmse.evaluate(predictions):.25f} £")
print(f"Mean Absolute Error:  {evaluator_mae.evaluate(predictions):.25f} £")
print(f"R²:   {evaluator_r2.evaluate(predictions):.2f}")

Root Mean Square Error: 56.9222476834416113433690043 £
Mean Absolute Error:  24.3010287869345624756078905 £
R²:   0.31


## Persisting the Model



In [18]:
model.write().save("../data/output/titanic_pipeline_model")

In [ ]:
spark.stop()